# Tidal Disruption Event (TDE) Classification

This notebook develops a machine learning classifier to detect **Tidal Disruption Events (TDEs)** from time-domain photometric observations.

A TDE occurs when a star passes sufficiently close to a supermassive black hole and is torn apart by tidal forces. These events produce distinctive multi-band light curves.

## Objective

The goal of this project is to:

- Extract meaningful features from raw time-series photometric data  
- Handle severe class imbalance (~5% positives)  
- Train a robust LightGBM classifier  
- Optimize the decision threshold to maximize F1 score  

The competition metric is **F1 score**, which balances precision and recall.

In [ ]:
import os
import numpy as np
import pandas as pd
from scipy.stats import skew, kurtosis

from sklearn.model_selection import GroupKFold
from sklearn.metrics import f1_score
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

import lightgbm as lgb
from catboost import CatBoostClassifier

import warnings
warnings.filterwarnings("ignore")

OSError: dlopen(/opt/miniconda3/envs/kaggle-mallorn/lib/python3.14/site-packages/lightgbm/lib/lib_lightgbm.dylib, 0x0006): Library not loaded: @rpath/libomp.dylib
  Referenced from: <D44045CD-B874-3A27-9A61-F131D99AACE4> /opt/miniconda3/envs/kaggle-mallorn/lib/python3.14/site-packages/lightgbm/lib/lib_lightgbm.dylib
  Reason: tried: '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/local/lib/libomp/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/local/lib/libomp/libomp.dylib' (no such file), '/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/homebrew/opt/libomp/lib/libomp.dylib' (no such file), '/opt/local/lib/libomp/libomp.dylib' (no such file), '/System/Volumes/Preboot/Cryptexes/OS/opt/local/lib/libomp/libomp.dylib' (no such file), '/opt/miniconda3/envs/kaggle-mallorn/lib/python3.14/lib-dynload/../..

In [ ]:
BASE_PATH = "../data"

## Feature Engineering

In [ ]:
def load_lightcurves(dataset_type="train"):
    """
    Combines all split lightcurve files into one cached file.
    """

    combined_dir = os.path.join(BASE_PATH, "combined_curves")
    os.makedirs(combined_dir, exist_ok=True)

    combined_path = os.path.join(
        combined_dir,
        f"{dataset_type}_all_full_lightcurves.csv"
    )

    if os.path.exists(combined_path):
        print(f"Using cached file: {combined_path}")
        return pd.read_csv(combined_path)

    print("Building combined lightcurve file...")

    log_path = os.path.join(BASE_PATH, f"{dataset_type}_log.csv")
    log_df = pd.read_csv(log_path)

    frames = []

    for split_name in log_df["split"].unique():

        filename = f"{dataset_type}_full_lightcurves.csv"
        path = os.path.join(BASE_PATH, split_name, filename)

        if os.path.exists(path):
            frames.append(pd.read_csv(path))

    if not frames:
        raise FileNotFoundError("No lightcurve files found.")

    combined = pd.concat(frames, ignore_index=True)
    combined.to_csv(combined_path, index=False)

    return combined


def extract_features(lc_df, meta_df=None):
    """
    Advanced feature engineering for TDE detection.
    Adds energy, baseline contrast, peak alignment, rest-frame corrections,
    and stronger cross-band features.
    """

    features = []

    # If metadata provided (for redshift)
    if meta_df is not None:
        meta_map = meta_df.set_index("object_id")
    else:
        meta_map = None

    for obj_id, group in lc_df.groupby("object_id"):

        obj_features = {"object_id": obj_id}

        # -----------------------------------
        # Global Behavior
        # -----------------------------------
        all_flux = group["Flux"].values
        all_time = group["Time (MJD)"].values

        obj_features["global_std"] = np.std(all_flux)
        obj_features["global_ptp"] = all_flux.max() - all_flux.min()
        obj_features["global_positive_frac"] = (all_flux > 0).mean()
        obj_features["global_negative_frac"] = (all_flux < 0).mean()

        # Integrated energy across all bands
        try:
            obj_features["global_energy"] = np.trapz(np.abs(all_flux), all_time)
        except:
            obj_features["global_energy"] = 0

        # -----------------------------------
        # Per-Band Features
        # -----------------------------------
        peak_times = []
        band_means = {}
        peak_dict = {}

        for f in ["u", "g", "r", "i", "z", "y"]:

            band = group[group["Filter"] == f]

            if len(band) < 5:
                continue

            flux = band["Flux"].values
            time = band["Time (MJD)"].values
            flux_err = band["Flux_err"].values

            prefix = f"band_{f}_"

            mean_flux = flux.mean()
            std_flux = flux.std()
            band_means[f] = mean_flux

            obj_features[prefix + "mean"] = mean_flux
            obj_features[prefix + "std"] = std_flux
            obj_features[prefix + "ptp"] = flux.max() - flux.min()
            obj_features[prefix + "skew"] = skew(flux)
            obj_features[prefix + "kurt"] = kurtosis(flux)

            obj_features[prefix + "p10"] = np.percentile(flux, 10)
            obj_features[prefix + "p25"] = np.percentile(flux, 25)
            obj_features[prefix + "p75"] = np.percentile(flux, 75)
            obj_features[prefix + "p90"] = np.percentile(flux, 90)

            obj_features[prefix + "iqr"] = (
                obj_features[prefix + "p75"] - obj_features[prefix + "p25"]
            )

            # Negative flux fraction
            obj_features[prefix + "neg_frac"] = (flux < 0).mean()

            # Signal-to-noise
            snr = flux / (flux_err + 1e-6)
            obj_features[prefix + "mean_snr"] = snr.mean()

            # Baseline (20th percentile)
            baseline = np.percentile(flux, 20)
            amplitude = flux.max() - baseline

            obj_features[prefix + "amplitude"] = amplitude
            obj_features[prefix + "amplitude_ratio"] = amplitude / (abs(baseline) + 1e-6)

            # Integrated energy per band
            try:
                obj_features[prefix + "energy"] = np.trapz(np.abs(flux), time)
            except:
                obj_features[prefix + "energy"] = 0

            # Peak time
            peak_idx = np.argmax(flux)
            peak_time = time[peak_idx]
            peak_times.append(peak_time)

            peak_dict[f] = peak_time

            obj_features[prefix + "duration"] = time.max() - time.min()

            before_peak = time < peak_time
            after_peak = time > peak_time

            # Rise slope
            slope_rise = 0
            if before_peak.sum() > 1:
                try:
                    slope_rise = np.polyfit(
                        time[before_peak] - time[before_peak].mean(),
                        flux[before_peak], 1
                    )[0]
                except:
                    pass

            # Decay slope
            slope_decay = 0
            if after_peak.sum() > 1:
                try:
                    slope_decay = np.polyfit(
                        time[after_peak] - time[after_peak].mean(),
                        flux[after_peak], 1
                    )[0]
                except:
                    pass

            obj_features[prefix + "rise_slope"] = slope_rise
            obj_features[prefix + "decay_slope"] = slope_decay
            obj_features[prefix + "asymmetry"] = slope_rise / (abs(slope_decay) + 1e-6)

            # Pre/Post peak contrast
            if before_peak.sum() > 0 and after_peak.sum() > 0:
                pre_mean = flux[before_peak].mean()
                post_mean = flux[after_peak].mean()
                obj_features[prefix + "pre_post_ratio"] = pre_mean / (post_mean + 1e-6)
            else:
                obj_features[prefix + "pre_post_ratio"] = 0

        # -----------------------------------
        # Cross-Band Features
        # -----------------------------------

        # Peak alignment (important!)
        if len(peak_times) > 1:
            obj_features["peak_time_std"] = np.std(peak_times)
        else:
            obj_features["peak_time_std"] = 0

        filters = list(peak_dict.keys())

        for i in range(len(filters)):
            for j in range(i+1, len(filters)):

                f1 = filters[i]
                f2 = filters[j]

                obj_features[f"lag_{f1}_{f2}"] = (
                    peak_dict[f1] - peak_dict[f2]
                )

        # Color features
        color_pairs = [("g", "r"), ("r", "i"), ("u", "g"), ("i", "z")]

        for b1, b2 in color_pairs:
            if b1 in band_means and b2 in band_means:
                obj_features[f"color_{b1}_{b2}"] = band_means[b1] - band_means[b2]

        # -----------------------------------
        # Rest-Frame Correction
        # -----------------------------------

        if meta_map is not None and obj_id in meta_map.index:
            z = meta_map.loc[obj_id]["Z"]
            duration = all_time.max() - all_time.min()
            obj_features["rest_duration"] = duration / (1 + z)
        else:
            obj_features["rest_duration"] = 0

        features.append(obj_features)

    return pd.DataFrame(features)


def load_features(dataset_type="train"):
    """
    Loads processed features or builds them if missing.
    """

    processed_path = os.path.join(
        BASE_PATH,
        f"{dataset_type}_features.csv"
    )

    if os.path.exists(processed_path):
        print(f"Using cached features: {processed_path}")
        df = pd.read_csv(processed_path)
    else:
        print("Extracting features...")

        lc_df = load_lightcurves(dataset_type)
        df = extract_features(lc_df)

        log_df = pd.read_csv(
            os.path.join(BASE_PATH, f"{dataset_type}_log.csv")
        )

        if dataset_type == "train":
            df = df.merge(
                log_df[["object_id", "target", "Z", "EBV"]],
                on="object_id",
                how="inner"
            )
        else:
            df = df.merge(
                log_df[["object_id", "Z", "EBV"]],
                on="object_id",
                how="inner"
            )

        df.to_csv(processed_path, index=False)

    return df


def get_prepared_dataset(dataset_type="train"):
    """
    Returns model-ready dataset.
    """

    df = load_features(dataset_type)

    if dataset_type == "train":
        X = df.drop(columns=["object_id", "target"])
        y = df["target"]
        return X, y

    elif dataset_type == "test":
        X = df.drop(columns=["object_id"])
        ids = df["object_id"]
        return X, ids

In [ ]:
X, y = get_prepared_dataset("train")

pos = y.sum()
neg = len(y) - pos

print(f"Total samples: {len(y)}")
print(f"Non-TDE (0): {neg}")
print(f"TDE (1): {pos}")
print(f"Positive rate: {pos/len(y):.4f}")

In [ ]:
pos_weight = (len(y) - y.sum()) / y.sum()
print("Scale pos weight:", pos_weight)

## Model Selection

In [ ]:
class DeepEnsemble:

    def __init__(self, scale_pos_weight=1.0, random_state=42):
        self.scale_pos_weight = scale_pos_weight
        self.random_state = random_state
        self.models = {}

    def _get_feature_subsets(self, X):

        temporal_cols = [c for c in X.columns if any(
            k in c for k in ["slope", "duration", "asymmetry"]
        )]

        statistical_cols = [c for c in X.columns if any(
            k in c for k in ["mean", "std", "skew", "kurt", "ptp"]
        )]

        return temporal_cols, statistical_cols

    def fit(self, X, y):

        temporal_cols, statistical_cols = self._get_feature_subsets(X)

        cb_params = dict(
            iterations=1200,
            depth=6,
            learning_rate=0.02,
            l2_leaf_reg=15,
            subsample=0.7,
            loss_function="Logloss",
            verbose=0,
            random_seed=self.random_state,
            scale_pos_weight=self.scale_pos_weight,
            allow_writing_files=False
        )

        # Base model (All features)
        self.models["base"] = CatBoostClassifier(**cb_params)
        self.models["base"].fit(X, y)

        # Temporal specialist
        if temporal_cols:
            self.models["temporal"] = CatBoostClassifier(**cb_params)
            self.models["temporal"].fit(X[temporal_cols], y)

        # Statistical specialist
        if statistical_cols:
            self.models["statistical"] = CatBoostClassifier(**cb_params)
            self.models["statistical"].fit(X[statistical_cols], y)

        # Support model: MLP
        self.models["mlp"] = Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler()),
            ("net", MLPClassifier(
                hidden_layer_sizes=(128, 64),
                alpha=0.01,
                max_iter=600,
                random_state=self.random_state
            ))
        ])
        self.models["mlp"].fit(X, y)

        # Support model: KNN
        self.models["knn"] = Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler()),
            ("knn", KNeighborsClassifier(
                n_neighbors=15,
                weights="distance"
            ))
        ])
        self.models["knn"].fit(X, y)

        return self

    def predict_proba(self, X):

        temporal_cols, statistical_cols = self._get_feature_subsets(X)

        p_base = self.models["base"].predict_proba(X)[:, 1]

        p_temporal = p_base
        if "temporal" in self.models and temporal_cols:
            p_temporal = self.models["temporal"].predict_proba(X[temporal_cols])[:, 1]

        p_stat = p_base
        if "statistical" in self.models and statistical_cols:
            p_stat = self.models["statistical"].predict_proba(X[statistical_cols])[:, 1]

        p_mlp = self.models["mlp"].predict_proba(X)[:, 1]
        p_knn = self.models["knn"].predict_proba(X)[:, 1]

        # Weighted blending
        final_prob = (
            0.45 * p_base +
            0.20 * p_temporal +
            0.15 * p_stat +
            0.10 * p_mlp +
            0.10 * p_knn
        )

        return final_prob

    def predict(self, X, threshold=0.5):
        probs = self.predict_proba(X)
        return (probs >= threshold).astype(int)

## Cross-Validation with OOF Generation

In [ ]:

gkf = GroupKFold(n_splits=5)

groups = X.index  # or object_id if you kept it

oof_preds = np.zeros(len(y))
models = []

for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups), 1):

    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    pos_weight = (len(y_train) - y_train.sum()) / y_train.sum()

    model = DeepEnsemble(scale_pos_weight=pos_weight)
    model.fit(X_train, y_train)

    val_probs = model.predict_proba(X_val)

    oof_preds[val_idx] = val_probs
    models.append(model)

    fold_f1 = f1_score(y_val, (val_probs >= 0.5).astype(int))
    print(f"Fold {fold} F1: {fold_f1:.4f}")

In [ ]:
best_f1 = 0
best_threshold = 0.5

for t in np.linspace(0.01, 0.5, 200):
    score = f1_score(y, (oof_preds >= t).astype(int))
    if score > best_f1:
        best_f1 = score
        best_threshold = t

print("Best OOF F1:", best_f1)
print("Best OOF threshold:", best_threshold)
print("Mean predicted probability:", oof_preds.mean())
print("True positive rate:", y.mean())

In [ ]:
print("OOF max:", oof_preds.max())
print("OOF mean:", oof_preds.mean())

## Final Model Training

After selecting hyperparameters and optimal threshold:

- We retrain the CatBoost-based ensemble on the full training dataset
- Use the validated configuration  
- Apply the optimized probability threshold for submission  

This ensures the submission model is consistent with cross-validation findings.

In [ ]:
X_test, test_ids = get_prepared_dataset("test")
X_test = X_test.reindex(columns=X.columns, fill_value=0)

test_preds = np.zeros(len(X_test))

for model in models:
    test_preds += model.predict_proba(X_test)

test_preds /= len(models)
test_binary = (test_preds >= best_threshold).astype(int)

print("Test max:", test_preds.max())
print("Test mean:", test_preds.mean())

In [ ]:
unique, counts = np.unique(test_binary, return_counts=True)
class_counts = dict(zip(unique, counts))

print("Prediction summary:")
print("Class 0 (Non-TDE):", class_counts.get(0, 0))
print("Class 1 (TDE):", class_counts.get(1, 0))
print("Total predictions:", len(test_binary))
print("Positive rate:", class_counts.get(1, 0) / len(test_binary))

In [ ]:
submission = pd.DataFrame({
    "object_id": test_ids,
    "prediction": test_binary
})

submission.to_csv("submission.csv", index=False)

print("Submission saved.")